In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy as sp
import scipy.stats as stats
import warnings as warn
%matplotlib qt

def bpsk_alt(x,e_vec,nbits,symbol_rate,wincut,fc,Ebit,fs=800e6):
    """
    nbits: number of bits to simulate
    x = the time index length/array 
    tbit = number of samples per symbol
    e_vec = the carrier waveform
    symbol_rate = symbol rate in ksps
    wincut = idk (used in firwin for cutoff)
    fc = carrier frequency
    Ebit = idk (energy per bit?)
    """
    #binary phase shift keyed
    tbit = int( fs / (symbol_rate*1e3) )
    #print('making index range...')
    x = np.arange(nbits*tbit)
    #print('making bit seq...')
    bit_seq = np.random.randint(0,2,size=(int(len(x)/tbit)+1,))
    bit_seq = (2*bit_seq)-1
    pulse = np.ones(tbit)
    #print('making symbol seq...')
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]
    
    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=wincut/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='fft')
    
    #apply carrier signal
    ts = 1/fs
    #print('making carrier signal...')
    e_vec = np.exp(2.j*np.pi*fc*x*ts)
    #e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    #print('modulating...')
    
    sig = sym_seq * e_vec
    return sig,sym_seq,bit_seq


In [2]:

signal = bpsk_alt(400, np.array([]), 2, 5e3, 1.0, 100e6, 1, fs=1000e6)
plt.plot(np.real(signal[0]), label='real')
plt.plot(np.imag(signal[0]), label='imaginary')
plt.title('BPSK Signal')

plt.show()
print(f'symbol sequence: {signal[1]}')
print(f'bit sequence: {signal[2]}')


symbol sequence: [ 5.00000000e-01  5.28638690e-01  5.57218524e-01  5.85680864e-01
  6.13967507e-01  6.42020899e-01  6.69784347e-01  6.97202227e-01
  7.24220189e-01  7.50785355e-01  7.76846506e-01  8.02354272e-01
  8.27261306e-01  8.51522449e-01  8.75094891e-01  8.97938315e-01
  9.20015038e-01  9.41290132e-01  9.61731539e-01  9.81310171e-01
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.0000

In [3]:
dft = np.fft.fft(signal[0])
plt.plot(np.abs(dft))
plt.show()

In [4]:
def qpsk(x,nbits,symbol_rate,wincut,fc,Ebit,fs=800e6):
    #quad phase shift keyed
    """
    x = the time index length/array 
    nbits: number of bits to simulate
    tbit = number of samples per symbol
    symbol_rate = symbol rate in ksps
    wincut = idk (used in firwin for cutoff)
    fc = carrier frequency
    Ebit = idk (energy per bit?)
    fs = sampling frequency
    """
    #derived number of samples per symbol (this should go inside each rfi generator)
    tbit = int( fs / (symbol_rate*1e3) )
    
    x = np.arange(nbits*tbit)
    bit_seq = np.random.randint(1,5,size=(int(len(x)/tbit)+1,))
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]


    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=1.0/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='auto')


    #apply carrier signal
    ts = 1/fs
    
    #theres a faster way to optimize qpsk i think but this is fine for overnight
    arg = (2.j*np.pi*fc*x*ts) + (1.j*(np.pi/4)*(2*sym_seq-1))
    sig = np.exp(arg)
    #e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    
    #sig = e_vec

    return sig,sym_seq,bit_seq

In [5]:
quadsig = qpsk(400, 2, 2e3, 1.0, 50e6, 1, fs=1000e6)
plt.close('all')
plt.plot(np.real(quadsig[0]), label='real')
plt.plot(np.imag(quadsig[0]), label='imaginary')
plt.title('QPSK Signal')
plt.legend()
plt.show()
print(f'symbol sequence: {quadsig[1]}')
print(f'bit sequence: {quadsig[2]}')


symbol sequence: [2.         2.04583472 2.09165437 2.13744387 2.18318817 2.22887225
 2.27448112 2.31999983 2.3654135  2.4107073  2.45586646 2.50087632
 2.54572227 2.59038984 2.63486462 2.67913235 2.72317887 2.76699016
 2.81055235 2.85385169 2.8968746  2.93960767 2.98203764 3.02415145
 3.0659362  3.10737922 3.148468   3.18919026 3.22953393 3.26948717
 3.30903834 3.34817606 3.38688919 3.42516682 3.4629983  3.50037325
 3.53728154 3.57371332 3.60965899 3.64510927 3.68005514 3.71448787
 3.74839903 3.78178048 3.81462439 3.84692323 3.8786698  3.90985719
 3.94047881 3.97052839 4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         

In [6]:
QpskDft = np.fft.fft(quadsig[0])
plt.plot(np.abs(QpskDft))
plt.show()

In [7]:
def ask_mod(x,e_vec,nsym,tbit,wincut,symbol_rate,fc,Ebit=0.0,N0=None,fs=800e6, biases= np.array([0.5, 1.0]) ):

    #ASK but with power levels between 0 and 1 to test dc stuff
    tbit = int( fs / (symbol_rate*1e3) )
    x= np.arange(nsym*tbit)
    symsz = int((len(x)/tbit))+1
    bit_seq = np.random.randint(0,len(biases),size=symsz).astype(np.float16)
    for i in range(len(biases)):
        bit_seq[bit_seq==i] = biases[i]
    # print(tbit)
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]


    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=wincut/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='fft')


    #apply carrier signal
    ts = 1/fs
    e_vec = np.exp(2.j*np.pi*fc*np.arange(len(sym_seq))*ts)
    # e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    # print(len(sym_seq))
    # print(len(e_vec))
    
    sig = sym_seq * e_vec

    return sig,sym_seq,bit_seq


In [8]:
askSig = ask_mod(400, np.array([]), 10, 5e3, 1.0, 5e3, 100e6, fs=1000e6, biases=np.array([0.2, 0.5, 0.8, 1.0]) )
plt.close('all')
plt.plot(np.real(askSig[0]), label='real')
plt.plot(np.imag(askSig[0]), label='imaginary')
plt.title('ASK Signal')
plt.legend()
plt.show()
print(f'symbol sequence: {askSig[1]}')
print(f'bit sequence: {askSig[2]}')

symbol sequence: [0.25       0.26431934 0.27860926 ... 0.4684303  0.44566599 0.4228077 ]
bit sequence: [0.5 1.  1.  1.  1.  0.2 1.  0.5 1.  0.8 1. ]


In [9]:
askDft = np.fft.fft(askSig[0])
plt.plot(np.abs(askDft))
plt.show()

In [10]:
def vco_complex(v_in,f0,K0,ts,tbit,nbits):
    """
    Takes an input voltage and returns a variable frequency waveform
    Inputs:
        v_in : input time-indexed voltages (1D)
        f0   : quiescent frequency of oscillator
        K0   : oscillator sensitivity (Hz / V)
        ts   : sampling time (reciprocal of sample rate)
    Output:
        v_out: output time-indexed voltages
    """
    #define stuff
    x = np.tile(np.arange(tbit),nbits)
    phase = np.empty(len(v_in))
    
    #define instantaneous frequencies and phases
    freq = f0 + K0*v_in
    phase = sp.integrate.cumulative_trapezoid(freq,dx=ts,initial=0)
    for i in range(nbits):
        phase[i*tbit:(i+1)*tbit] = phase[i*tbit]
        
    #create waveform
    arg = (2.j*np.pi*x*freq*ts) + 1.j*phase
    v_out = np.exp(arg)
    
    return v_out,freq,phase

#binary freq-shift keying - switch between 2 freqs
def bfsk(nbits,symbol_rate,f0,f1,Ebit=0.0,fs=800e6, N0=None):

    tbit = int( fs / (symbol_rate*1e3) )
    
    #make bit sequence (and turn into seq of +/- 0.5's for VCO)
    bit_seq = np.random.randint(0,high=2,size=nbits)
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse) - 0.5
    
    #lo-pass filter
    hann = np.hanning(int(tbit*0.2))
    # sym_seq = np.convolve(sym_seq,hann,mode='same')
    # sym_seq = sym_seq/(2*np.max(sym_seq))
    
    ts=1/fs
    Ebit_linear = 10**(Ebit/10.0)

    #define VCO f0,K0 based on inputs
    vco_center = (f0+f1)/2
    #assuming sym_seq = 1 corresponds to voltage = 1V
    vco_sens = (f1-f0)
    
    sig,f,p = vco_complex(sym_seq, vco_center, vco_sens, ts, tbit, nbits)

    sig *= np.sqrt(Ebit_linear)

    return sig,sym_seq,f,p


In [11]:
bfskSig = bfsk(3, 5e3, 100e6, 35e6, fs=1000e6)
plt.close('all')
plt.plot(np.real(bfskSig[0]), label='real')
plt.plot(np.imag(bfskSig[0]), label='imaginary')
plt.title('BFSK Signal')
plt.legend()
plt.show()

print(f'symbol sequence: {bfskSig[1][:20]}')
print(f'frequency from VCO: {bfskSig[2][:20]}')
print(f'phase from VCO: {bfskSig[3][:20]}')


symbol sequence: [-0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5
 -0.5 -0.5 -0.5 -0.5 -0.5 -0.5]
frequency from VCO: [1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08
 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08 1.e+08]
phase from VCO: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [12]:
bfskDft = np.fft.fft(bfskSig[0])
plt.plot(np.abs(bfskDft))
plt.show()

In [13]:
def mfrg(Rmethod, nbits, symbol_rate, fc, biases = np.array([0.5,1.0]), f0 = 100e6, f1 = 75e6, Ebit = 1.0, fs = 800e6, nchans = 2048, wincut = 1.0):
    """
    Parameters:
    Rmethod: modulation method to use (e.g. 'ask', 'bpsk', etc.)
    nbits: number of bits to simulate
    symbol_rate: symbol rate in ksps
    fc: center carrier frequency
    biases: for ask_mod, the power levels to use (between 0 and 1)
    f0: for bfsk, the lower frequency
    f1: for bfsk, the higher frequency
    Ebit: energy per bit
    fs: sampling frequency
    nchans: number of channels to simulate (for multi-channel RFI) - centered at fc and seperated by 1 KHz
    wincut: window cut-off frequency

    Returns: a 2D array of shape (nchans, nbits*tbit) containing a simulated RFI signal for each channel using the chosen method
    """
    #derived number of samples per symbol (this should go inside each rfi generator)
    Rmethod = Rmethod.lower().strip()
    tbit = int( fs / (symbol_rate*1e3) )
    # #index time array
    x = np.arange(nbits*tbit)   
    sigarray = np.empty((nchans, len(x)), dtype=np.complex128)
    # print(sigarray.shape)
    
    # pulse = np.ones(tbit)

    if Rmethod == 'ask':
        for i in range(nchans):
            offest = (i-(nchans//2))*1e5
            fc_chan = fc + offest
                
            sig = ask_mod(0, np.array([]), nbits, 0, wincut, symbol_rate, fc_chan, Ebit, None, fs, biases)[0]

            sigarray[i,:] = sig

    elif Rmethod == 'bpsk':
        for i in range(nchans):
            offest = (i-(nchans/2))*1e5
            fc_chan = fc + offest

            sig = bpsk_alt(0, np.array([]), nbits, symbol_rate, wincut, fc_chan, Ebit, fs)[0]

            sigarray[i,:] = sig

    elif Rmethod == 'qpsk':
        for i in range(nchans):
            offest = (i-(nchans//2))*1e5
            fc_chan = fc + offest
            
            sig = qpsk(0, nbits, symbol_rate, wincut, fc_chan, Ebit, fs)[0]

            sigarray[i,:] = sig
    
    #This one is special and requires VCO implementation. 
    elif Rmethod == 'bfsk':
        for i in range(nchans):
            offest = (i-(nchans/2))*1e5
            f1_chan = f1 + offest
            f0_chan = f0 + offest

            sig = bfsk(nbits, symbol_rate, f0_chan, f1_chan, Ebit, fs=fs)[0]

            sigarray[i,:] = sig
    else:
        raise ValueError('Invalid Rmethod. Must be one of: ask, bpsk, qpsk, bfsk')

    return sigarray
     

In [14]:
MultiSig = mfrg('bpsk', 9, 5e3, 150e6, biases= np.array([0.5, 1]), f1=350e6, f0=150e6, wincut=39, fs=1000e6, nchans=4)
time = np.arange(MultiSig.shape[1])
# MultiSig 
print(MultiSig.shape)
noise = np.random.normal(0, 0.1, size=np.shape(MultiSig))
print(MultiSig.shape)
# MultiSig += noise

MultiDFT = np.fft.fft(MultiSig, axis=1)

#pseudo decibel conversion
MDFT_linear = np.abs(MultiDFT)**2
MDFT_db = 10*np.log10(MDFT_linear)

MultiSig *= np.log10(np.arange(MultiSig.shape[1]))/np.log10(MultiSig.shape[1])


# plt.close('all')
# plt.title('multi frequency qpsk Signal')
# plt.xlabel('temporal frequency')
# plt.ylabel('Frequency ')
# plt.colorbar(label='Magnitude (dB)')
# plt.plot(np.real(MultiSig[:,500]), label='real')
# plt.plot(np.imag(MultiSig[:,2000]), label='imaginary')

# # plt.plot(np.abs(MDFT_db[2000]))
# # plt.plot(np.abs(MDFT_db[200]))

# plt.show()
# plt.imshow(np.abs(MDFT_db))

# print(f'symbol sequence: {MultiSig[1][:20]}')
# print(f'bit sequence: {MultiSig[2][:20]}')
# print(f'phase from VCO: {qpskSig[3][:20]}')

# print()

(4, 1800)
(4, 1800)


/tmp/ipykernel_729545/3140718414.py:15: RuntimeWarning: divide by zero encountered in log10
  MultiSig *= np.log10(np.arange(MultiSig.shape[1]))/np.log10(MultiSig.shape[1])
/tmp/ipykernel_729545/3140718414.py:15: RuntimeWarning: invalid value encountered in multiply
  MultiSig *= np.log10(np.arange(MultiSig.shape[1]))/np.log10(MultiSig.shape[1])


In [15]:
# plt.close('all')
# print(MDFT_db.shape)
# spectra = np.sum(np.abs(MDFT_db), axis=1)
# plt.plot(spectra)
# plt.title('BFSK clean Summed DFT across temporal')
# plt.xlabel('Frequency index')
# plt.ylabel('Magnitude')
# plt.show()
# plt.savefig('plots/BFSK_spectra_clean.png')

In [16]:
#generate all the plots

plt.close('all')
plt.plot(np.real(MultiSig[0]), label='real', linewidth=0.75)
plt.plot(np.imag(MultiSig[0]), label='imaginary', linewidth=0.75)
plt.title('bfsk Signal')
plt.xlabel('Time index')
plt.ylabel('Amplitude')
plt.legend()
plt.show()
# plt.savefig('plots/BFSK_signal_clean.png')

plt.close('all')
plt.plot(np.abs(MDFT_db[2]), label='channel 2000')
plt.plot(np.abs(MDFT_db[3]), label='channel 200')
plt.title('DFT for two channels')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.legend()
plt.show()
# plt.savefig('plots/BFSK_dft_two_channels_clean.png')

plt.close('all')
plt.imshow(np.abs(MDFT_db))
plt.title('DFT for all channels')
plt.xlabel('Frequency index')
plt.ylabel('Time index')
plt.colorbar(label='Magnitude (dB)' )
plt.show()
# plt.savefig('plots/BFSK_dft_waterfall_clean.png')

plt.close('all')
print(MDFT_db.shape)
spectra = np.sum(np.abs(MDFT_db), axis=0)
plt.plot(spectra, linewidth=0.5)
plt.title('BFSK clean Summed DFT across temporal')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.show()
# plt.savefig('plots/BFSK_spectra_clean.png')


(4, 1800)


In [17]:
##Spectral Kurtosis##
1800 %258

252

In [18]:
def SKest (data, N =1 , d =1, m=256):
    """
    data: 2d Array (nchans, nsamples)
    N: number of averaged channels (default 1)
    d = shape factor (default 1)
    m: size of time bins to average over for SK estimation
    """

    bins = data.shape[1]//m
    print(data.shape[0])
    print(bins)
    SKarr = np.empty((data.shape[0], bins))

    for i in range(bins):
        
        S1 = np.sum(np.abs(data[:,(i*m):(i+1)*m])**2, axis=1)
        S2 = np.sum(np.abs(data[:,(i*m):(i+1)*m])**4, axis=1)
        SK = (m*N*d+1)/(m-1) * (m*S2/S1**2 - 1)
        SKarr[:, i] = SK

    #output 2D array of shape (nchans, bins) containing SK values for each bin 
    return SKarr

def ms_SKest (data, n =1 , d =1, m=256):
    
    Tbins = data.shape[1]//m
    Cbins = data.shape[0]//n
    SKarr = np.empty((Cbins, Tbins))

    for i in range(Tbins):
        
        S1 = np.sum(np.abs(data[:,(i*m):(i+1)*m])**2, axis=1)
        S2 = np.sum(np.abs(data[:,(i*m):(i+1)*m])**4, axis=1)

        for j in range(Cbins):
            S12 = np.sum(S1[j*n:(j+1)*n])
            S22 = np.sum(S2[j*n:(j+1)*n])

            SK = (m*n*d+1)/(m-1) * (m*S22/S12**2 - 1)
            SKarr[j, i] = SK
    return SKarr   

def SK_thresholds (m, n=1, d=1, p = 0.00013499):
    """
    m: Size of time bins to average over for SK estimation 
    n: number of averaged channels
    d: shape factor (default 1)
    p: desired probability of false alarm (default 0.00013499 for 5 sigma)

    returns: lower and upper SK thresholds for given parameters
    """
    x = np.random.normal(0, 1, size=[2048, m]) + 1j*np.random.normal(0, 1, size=[2048, m])
    SKx = SKest(x, N=n, d=d,m=m)
    # skew = np.mean(SKx)*m / ((m-1)*(m-2)*np.std(SKx)**3)

    realX = SKx.real
    imagX= SKx.imag

    skewR, locR, scaleR = scipy.stats.pearson3.fit(realX)
    skewI, locI, scaleI = scipy.stats.pearson3.fit(imagX)

    lower_threshR = scipy.stats.pearson3.ppf(p, skewR, locR, scaleR)
    upper_threshR = scipy.stats.pearson3.ppf(1 - p, skewR, locR, scaleR)
    # lower_threshI = scipy.stats.pearson3.ppf(p, skewI, locI, scaleI)
    # upper_threshI = scipy.stats.pearson3.ppf(1 - p, skewI, locI, scaleI)

    return lower_threshR, upper_threshR

def ms_SK_thresholds (m, n=1, d=1, p = 0.00013499):
    """
    m: Size of time bins to average over for SK estimation 
    n: number of averaged channels
    d: shape factor (default 1)
    p: desired probability of false alarm (default 0.00013499 for 5 sigma)
    """
    x = np.random.normal(0, 1, size=[2048, m]) + 1j*np.random.normal(0, 1, size=[2048, m])
    SKx = ms_SKest(x, n=n, d=d,m=m)
    # skew = np.mean(SKx)*m / ((m-1)*(m-2)*np.std(SKx)**3)

    realX = SKx.real
    imagX= SKx.imag

    skewR, locR, scaleR = scipy.stats.pearson3.fit(realX)
    skewI, locI, scaleI = scipy.stats.pearson3.fit(imagX)

    lower_threshR = scipy.stats.pearson3.ppf(p, skewR, locR, scaleR)
    upper_threshR = scipy.stats.pearson3.ppf(1 - p, skewR, locR, scaleR)
    # lower_threshI = scipy.stats.pearson3.ppf(p, skewI, locI, scaleI)
    # upper_threshI = scipy.stats.pearson3.ppf(1 - p, skewI, locI, scaleI)
    return lower_threshR, upper_threshR


In [19]:
#ConvRFI implementation
import torch

from ConvRFI  import  RFIconv,init_RFIconv
#This is a slightly modified version of the example 
def conv_det(data, agg_factor = [5,5,3,5]):
  #data must be real and in float32 format
  #aggression factor is sensitivity. [Vertical RFI, Horizontal RFI, left/right edges, top/bottom edges]

  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  if device!='cpu':
      torch.cuda.empty_cache()

  net = RFIconv(device=device)

  net = init_RFIconv(net,aggressive_factor = agg_factor,device=device).to(device)
  with torch.no_grad():
    output = net(torch.tensor(data.squeeze()[None,None,:,:]).to(device)).squeeze().cpu().numpy()
  data_test_ma = np.ma.masked_where((output),data.squeeze())

  if device!='cpu':
    torch.cuda.empty_cache()

  return data_test_ma 

In [20]:


#make pure noise test signal also the bakground noise for RFI signal testing
Noise = np.random.normal(0, 0.5, size=[256, 1820]) + 1.j*np.random.normal(0, 0.5, size=[256, 1820])

#adding sidelobes to signal. using a gaussian distrubtion to create sidelobes
gausNorm = np.clip(np.sort(np.abs(np.random.normal(0, 0.4, size=[1,1])),axis=0), 0, 1)
sidelobes = MultiSig[0:10,:] * gausNorm
lobeSig = np.concatenate((sidelobes, MultiSig, sidelobes[::-1]), axis=0)

#Add RFI signal to Noise signal
SigTest = Noise.copy()
sigSlice = slice(Noise.shape[0]//2 - MultiSig.shape[0]//2, Noise.shape[0]//2 + MultiSig.shape[0]//2), slice(-MultiSig.shape[1], None) #slice in the center of the channels pushed to the end of the time axis
SigTest[sigSlice] += MultiSig

print(SigTest.shape)

#Calc Sk vals
test = SKest(SigTest, N=1, d=1, m=67)

#Plotting SK estimates
# print(test) 
plt.close('all')
plt.plot(test, label='SK estimate')
# plt.scatter(range(len(test)), test)
# plt.scatter(range(len(sciTest)), sciTest, label='scipy kurtosis')
plt.xlabel('Channel index')
plt.ylabel('SK Estimate')
plt.show()
# plt.savefig('plots/BPSK_SK_estimate_clean.png')

(256, 1820)
256
27


/tmp/ipykernel_729545/1005719598.py:18: RuntimeWarning: invalid value encountered in divide
  SK = (m*N*d+1)/(m-1) * (m*S2/S1**2 - 1)


In [21]:
# Noise = np.random.normal(0, 0.5, size=[2048, 1800]) + 1.j*np.random.normal(0, 0.5, size=[2048, 1800])
# SKvals = SKest(Noise, N=1, d=1, bins=7)
#SK masking
RFiMit =SigTest.copy()
print(RFiMit.shape)
lower, upper= SK_thresholds(257,1,1)
print(lower, upper)
mask = np.kron(test, np.ones(RFiMit.shape[1]//7))
print(mask.shape)
# RFiMit[(mask < lower)|(mask > upper)] = 0

#ConvRFI call and masking
ConvMit = conv_det(np.float32(np.abs(SigTest)**2), agg_factor=[0.3,1.6,0.2,0.5])
print(ConvMit.shape)
RFIConv = SigTest.copy()
RFIConv[ConvMit.mask] = 0


#Plotting
plt.close('all')

fig, axs = plt.subplots(2, 2, figsize=(8, 8))
axs[0,0].imshow(np.abs(RFiMit), cmap="hot")

axs[0,0].set_title('SK Mitigated BPSK log with gaussian sidelobes with SNR of 2')
axs[0,0].set_xlabel('Time index')
axs[0,0].set_ylabel('Channel index')

axs[0,1].imshow(np.abs(SigTest), cmap="hot")
axs[0,1].set_title('Original BPSK log with gaussian sidelobes with SNR of 2')
axs[1,0].set_xlabel('Time index')
axs[1,0].set_ylabel('Channel index')

axs[1,1].imshow(np.abs(RFIConv), cmap="hot")
axs[1,1].set_title('Conv Mitigated BPSK log with gaussian sidelobes with SNR of 2')
axs[1,1].set_xlabel('Time index')
axs[1,1].set_ylabel('Channel index')    
plt.show()

#error rate calculation
#SK
errorCalc = RFiMit.copy()
errorCalc[errorCalc!=0] = 1
RFI = errorCalc[sigSlice]
flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
print(f'Flagged RFI percentage: {flagged}')
errorCalc[sigSlice] = 2
FPrate = np.count_nonzero(errorCalc == 0) / np.prod(errorCalc.shape)
print(f'False positive rate: {FPrate}')

#ConvRFI
errorCalc = RFIConv.copy()
errorCalc[errorCalc!=0] = 1
RFI = errorCalc[sigSlice]
flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
print(f'Flagged RFI percentage: {flagged}')
errorCalc[sigSlice] = 2
FPrate = np.count_nonzero(errorCalc == 0) / np.prod(errorCalc.shape)
print(f'False positive rate: {FPrate}')



(256, 1820)
2048
1
0.6800897746405016 1.5748857765529298
(256, 7020)
(256, 1820)
Flagged RFI percentage: 0.0
False positive rate: 0.0
Flagged RFI percentage: 0.67
False positive rate: 0.5960293612637363


In [22]:
#Defining high level functions to generate signals and test mitigation methods
def sigGen (Rmethod, nbits, symbol_rate, fc, biases = np.array([0.5,1.0]), f0 = 100e6, f1 = 75e6, Ebit = 1.0, fs = 100e6, wincut = 39, RFIchan = 16, effDC =1, SNR = 5, nchans = 64, add_sidelobes = False, sidelobe_size = 10):
    """
    Parameters:
    Rmethod: modulation method to use (e.g. 'ask', 'bpsk', etc.)
    nbits: number of bits to simulate
    symbol_rate: symbol rate in ksps
    fc: center carrier frequency
    biases: for ask_mod, the power levels to use (between 0 and 1)
    f0: for bfsk, the lower frequency
    f1: for bfsk, the higher frequency
    Ebit: energy per bit
    fs: sampling frequency
    wincut: window cut-off frequency
    RFIchan: number of RFI channels
    SNR: signal-to-noise ratio
    nchans: number of channels
    add_sidelobes: whether to add sidelobes
    sidelobe_size: size of sidelobes

    Returns: a 1D array containing a simulated RFI signal using the chosen method
    """
    RFIsig = mfrg(Rmethod, nbits, symbol_rate, fc, biases=biases, f1=f1, f0=f0, wincut=wincut, Ebit=Ebit, fs=1000e6, nchans=RFIchan)
    
    MultiDFT = np.fft.fft(RFIsig, axis=1)
    #pseudo decibel conversion
    MDFT_linear = np.abs(MultiDFT)**2
    MDFT_db = 10*np.log10(MDFT_linear)
    #Make RFI signal increase logly across time to test SK and ConvRFI performance on various signal levels
    RFIsig *= np.log10(np.arange(RFIsig.shape[1]))/np.log10(RFIsig.shape[1])

    #adding sidelobes to signal. using a gaussian distrubtion to create sidelobes
    if add_sidelobes:
        gausNorm = np.clip(np.sort(np.abs(np.random.normal(0, 0.4, size=[sidelobe_size,1])),axis=0), 0, 1)
        sidelobes = RFIsig[0:sidelobe_size,:] * gausNorm
        RFIsig = np.concatenate((sidelobes, RFIsig, sidelobes[::-1]), axis=0)

    #generate noise
    Noise = np.random.normal(0, 1/SNR, size=[nchans, int(RFIsig.shape[1]/effDC)]) + 1j*np.random.normal(0, 1/SNR, size=[nchans, int(RFIsig.shape[1]/effDC)])

    #Add RFI signal to Noise signal
    Signal = Noise.copy()
    sigSlice = slice(Noise.shape[0]//2 - RFIsig.shape[0]//2, Noise.shape[0]//2 + RFIsig.shape[0]//2), slice(-RFIsig.shape[1], None) #slice in the center of the channels pushed to the end of the time axis
    Signal[sigSlice] += RFIsig


    
    return Signal, MDFT_db, sigSlice

def SKmitigate (Signal, n=1, d=1, m=256):
    """
    Signal: 2D array of shape (nchans, nsamples) containing the signal to mitigate
    n: number of averaged channels
    d: shape factor (default 1)
    m: size of time bins to average over for SK estimation

    Returns: a 2D array of the same shape as Signal with RFI mitigated using spectral kurtosis
    """
        #make sure M is compatible with the temporal length of the data
    time = Signal.shape[1]
    if time % m != 0:
                
        print(f'M (time bin size) must be a factor of the number of time samples. Got M={m} and time samples={time}.')
        if m > time:
            m = time
        while time % m != 0:
            m += 1
        print(f'Adjusting M to {m} for compatibility.')
    print(f'M: {m}')
    
    SKvals = SKest(Signal, N=n, d=d, m=m)
    lower, upper = SK_thresholds(m, n, d)
    mask = np.kron(SKvals, np.ones(m))
    mitigated = Signal.copy()
    mitigated[(mask < lower)|(mask > upper)] = 0

    return mitigated

def ms_SKmitigate (Signal, n=1, d=1, m=256):
    """
    Multi-scale spectral kurtosis mitigation. Applies SK mitigation at multiple time scales and combines masks.

    Signal: 2D array of shape (nchans, nsamples) containing the signal to mitigate
    n: number of averaged channels for ms-SK (default 1)
    d: shape factor (default 1)
    m: size of time bins to average over for SK estimation

    Returns: a 2D array of the same shape as Signal with RFI mitigated using multi-scale spectral kurtosis
"""
    #make sure M is compatible with the temporal length of the data
    time = Signal.shape[1]
    if time % m != 0:
            
        print(f'M (time bin size) must be a factor of the number of time samples. Got M={m} and time samples={time}.')
        if m > time:
            m = time
        while time % m != 0:
            m += 1
        print(f'Adjusting M to {m} for compatibility.')
    print(f'M: {m}')

    #make sure N is compatible with the number of channels
    chans = Signal.shape[0]
    if chans % n != 0:
        print(f'N (number of averaged channels) must be a factor of the number of channels. Got N={n} and channels={chans}.')
        if n > chans:
            n = chans
        while chans % n != 0:
            n += 1
        print(f'Adjusting N to {n} for compatibility.')

    mitigated = Signal.copy()
    

    SKarr = ms_SKest(Signal, n=n, d=d, m=m)
    #output 2D array of shape (nchans, bins) containing SK values for each bin 
    mask = np.kron(SKarr, np.ones(m))
    mask = np.repeat(mask, n, axis=0)[:Signal.shape[0], :Signal.shape[1]]
    lower, upper = ms_SK_thresholds(m, n, d)
    # lower += -3
    # upper += -3
    print(f'ms-SK thresholds: lower={lower}, upper={upper}')
    mitigated[(mask < lower)|(mask > upper)] = 0


    return mitigated, SKarr

def ConvRFI_mitigate (Signal, agg_factor = [5,5,3,5], bins = 48):
    """Signal: 2D array of shape (nchans, nsamples) containing the signal to mitigate 
    agg_factor: aggression factor for ConvRFI. [Vertical RFI, Horizontal RFI, left/right edges, top/bottom edges]
    Returns: a 2D array of the same shape as Signal with RFI mitigated using ConvRFI
    """
    # bins = 50
    bins = int(bins)
    time = Signal.shape[1]
    if time % bins != 0:
            
        print(f'M (time bin size) must be a factor of the number of time samples. Got bins={bins} and time samples={time}.')
        if bins > time:
            bins = time
        while time % bins != 0:
            bins += 1
        print(f'Adjusting M to {bins} for compatibility.')
    print(f'M: {bins}')
    
    TAsig = np.empty((Signal.shape[0], bins))
    for i in range(bins):
        TAsig[:,i] = np.mean(Signal[:, i*bins:(i+1)*bins], axis=1)

    mitigated = conv_det(np.float32(np.abs(TAsig)**2), agg_factor=agg_factor)
    mitigated = np.kron(mitigated, np.ones(Signal.shape[1]//bins))
    output = Signal.copy()
    output[~mitigated.mask] = 0
    return output




In [23]:
Sig, SigDb, sigSlice = sigGen('bpsk', 9, 5e3, 150e6, biases= np.array([0.5, 1]), f1=350e6, f0=150e6, wincut=39, fs=1000e6, RFIchan=256, effDC = 0.5, SNR=2, nchans=2048, add_sidelobes=True, sidelobe_size=100)
SKmit = SKmitigate(Sig, n=1, d=1, m=257)
ms_SKmit, SKarr = ms_SKmitigate(Sig, n=4, d=1, m=257) 
ConvMit = ConvRFI_mitigate(Sig, agg_factor=[5,5,0.01,0.01], bins = 45)

plt.plot(SKarr, label='SK estimate')
plt.show()
# plt.close('all')

# msNoise = np.random.normal(0, 1, size=[2048, 257]) + 1j*np.random.normal(0, 1, size=[2048, 257])
# NoiseSK = ms_SKest(msNoise, n=4, d=1, m=257)
# plt.plot(NoiseSK, label='SK estimate for noise')
# plt.show()

#Plotting#Plotting

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('BPSK RFI Mitigation with SNR of 2 and Gaussian Sidelobes')
axs[0,0].imshow(np.abs(SKmit), cmap="hot")
axs[0,0].set_title('SK Mitigated ')
axs[0,0].set_xlabel('Time index')
axs[0,0].set_ylabel('Channel index')

axs[0,1].imshow(np.abs(Sig), cmap="hot")
axs[0,1].set_title('Original ')
axs[0,1].set_xlabel('Time index')
axs[0,1].set_ylabel('Channel index')

axs[1,0].imshow(np.abs(ms_SKmit), cmap="hot")
axs[1,0].set_title('Multi-Scale SK Mitigated ')
axs[1,0].set_xlabel('Time index')
axs[1,0].set_ylabel('Channel index')

axs[1,1].imshow(np.abs(ConvMit), cmap="hot")
axs[1,1].set_title('Conv Mitigated ')
axs[1,1].set_xlabel('Time index')
axs[1,1].set_ylabel('Channel index')    
plt.show()

#error rate calculation
#SK
errorCalc = SKmit.copy()
errorCalc[errorCalc!=0] = 1
RFI = errorCalc[sigSlice]
flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
print(f'Flagged RFI percentage: {flagged}')
errorCalc[sigSlice] = 2
FPrate = np.count_nonzero(errorCalc == 0) / np.prod(errorCalc.shape)
print(f'False positive rate: {FPrate}')
FN = 1-flagged
TP = flagged
FP = FPrate
TN = 1-FPrate
print(f'TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}')
print (f'Precision: {TP/(TP+FP)}, Recall: {TP/(TP+FN)}, Accuracy Score: {TP/(TP+FP+FN)}')

#ms_Sk
errorCalc = ms_SKmit.copy()
errorCalc[errorCalc!=0] = 1
RFI = errorCalc[sigSlice]
flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
print(f'Flagged RFI percentage: {flagged}')
errorCalc[sigSlice] = 2
FPrate = np.count_nonzero(errorCalc == 0) / np.prod(errorCalc.shape)
print(f'False positive rate: {FPrate}')
FN = 1-flagged
TP = flagged
FP = FPrate
TN = 1-FPrate
print(f'TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}')
print (f'Precision: {TP/(TP+FP)}, Recall: {TP/(TP+FN)}, Accuracy Score: {TP/(TP+FP+FN)}')

#ConvRFI
print("ConvRFI Error Calculation:")
errorCalc = ConvMit.copy()
errorCalc[errorCalc!=0] = 1
RFI = errorCalc[sigSlice]
flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
print(f'Flagged RFI percentage: {flagged}')
errorCalc[sigSlice] = 2
FPrate = np.count_nonzero(errorCalc == 0) / (np.prod(errorCalc.shape)-np.prod(RFI.shape))
print(f'False positive rate: {FPrate}')
FN = 1-flagged 
TP = flagged
FP = FPrate
TN = 1-FPrate
print(f'TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}')
print (f'Precision: {TP/(TP+FP)}, Recall: {TP/(TP+FN)}, Accuracy Score: {TP/(TP+FP+FN)}')

/tmp/ipykernel_729545/1165083991.py:30: RuntimeWarning: divide by zero encountered in log10
  RFIsig *= np.log10(np.arange(RFIsig.shape[1]))/np.log10(RFIsig.shape[1])
/tmp/ipykernel_729545/1165083991.py:30: RuntimeWarning: invalid value encountered in multiply
  RFIsig *= np.log10(np.arange(RFIsig.shape[1]))/np.log10(RFIsig.shape[1])


M (time bin size) must be a factor of the number of time samples. Got M=257 and time samples=3600.
Adjusting M to 300 for compatibility.
M: 300
2048
12


/tmp/ipykernel_729545/1005719598.py:18: RuntimeWarning: invalid value encountered in divide
  SK = (m*N*d+1)/(m-1) * (m*S2/S1**2 - 1)


2048
1
M (time bin size) must be a factor of the number of time samples. Got M=257 and time samples=3600.
Adjusting M to 300 for compatibility.
M: 300


/tmp/ipykernel_729545/1005719598.py:39: RuntimeWarning: invalid value encountered in scalar divide
  SK = (m*n*d+1)/(m-1) * (m*S22/S12**2 - 1)


ms-SK thresholds: lower=-2.217507957988422, upper=-1.7844957421644612
M: 45


/tmp/ipykernel_729545/1165083991.py:148: ComplexWarning: Casting complex values to real discards the imaginary part
  TAsig[:,i] = np.mean(Signal[:, i*bins:(i+1)*bins], axis=1)
/users/amuthiya/Sim-RFi/myenv/lib64/python3.11/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Flagged RFI percentage: 0.41154970760233917
False positive rate: 0.0003662109375
TP: 0.41154970760233917, TN: 0.9996337890625, FP: 0.0003662109375, FN: 0.5884502923976609
Precision: 0.999110957064252, Recall: 0.41154970760233917, Accuracy Score: 0.41139904877100214
Flagged RFI percentage: 0.5131578947368421
False positive rate: 0.00048828125
TP: 0.5131578947368421, TN: 0.99951171875, FP: 0.00048828125, FN: 0.48684210526315785
Precision: 0.9990493820983639, Recall: 0.5131578947368421, Accuracy Score: 0.5129074516452186
ConvRFI Error Calculation:
Flagged RFI percentage: 0.06812865497076023
False positive rate: 0.004358974358974359
TP: 0.06812865497076023, TN: 0.9956410256410256, FP: 0.004358974358974359, FN: 0.9318713450292397
Precision: 0.9398659550701253, Recall: 0.06812865497076023, Accuracy Score: 0.06783297278171176


In [24]:
def ConvAcc (Sig, agg_factor=[1.6,1.6,0.01,0.01], bins=50):
    ConvMit = ConvRFI_mitigate(Sig, agg_factor=agg_factor, bins=bins)
    errorCalc = ConvMit.copy()
    errorCalc[errorCalc!=0] = 1
    RFI = errorCalc[sigSlice]
    flagged = np.count_nonzero(RFI==0)/np.prod(RFI.shape)
    errorCalc[sigSlice] = 2
    FPrate = np.count_nonzero(errorCalc == 0) / (np.prod(errorCalc.shape)-np.prod(RFI.shape))
    FN = 1-flagged 
    TP = flagged
    FP = FPrate
    TN = 1-FPrate
    return TP/FP

scipy.optimize.minimize(lambda x: -ConvAcc(Sig, agg_factor=[x[0],x[1],x[2],x[3]], bins=x[4]), x0=[1.6,1.6,0.01,0.01, 50], bounds=[(0.01, 5), (0.1, 5), (0.001, 1.1), (0.001, 1.1), (10, 100)], method='Nelder-Mead')

M: 50


/tmp/ipykernel_729545/1165083991.py:148: ComplexWarning: Casting complex values to real discards the imaginary part
  TAsig[:,i] = np.mean(Signal[:, i*bins:(i+1)*bins], axis=1)


M: 50
M: 50
M: 50
M: 50
M (time bin size) must be a factor of the number of time samples. Got bins=52 and time samples=3600.
Adjusting M to 60 for compatibility.
M: 60
M (time bin size) must be a factor of the number of time samples. Got bins=47 and time samples=3600.
Adjusting M to 48 for compatibility.
M: 48
M (time bin size) must be a factor of the number of time samples. Got bins=49 and time samples=3600.
Adjusting M to 50 for compatibility.
M: 50
M: 48
M (time bin size) must be a factor of the number of time samples. Got bins=47 and time samples=3600.
Adjusting M to 48 for compatibility.
M: 48
M (time bin size) must be a factor of the number of time samples. Got bins=47 and time samples=3600.
Adjusting M to 48 for compatibility.
M: 48
M (time bin size) must be a factor of the number of time samples. Got bins=46 and time samples=3600.
Adjusting M to 48 for compatibility.
M: 48
M (time bin size) must be a factor of the number of time samples. Got bins=46 and time samples=3600.
Adjus

KeyboardInterrupt: 

In [ ]:
import sys 

sorted([(name, sys.getsizeof(obj))for name, obj in globals().items()], key=lambda x: x[1], reverse=True)

[('ConvMit', 117964928),
 ('errorCalc', 117964928),
 ('Sig', 117964928),
 ('SKmit', 117964928),
 ('ms_SKmit', 117964928),
 ('Noise', 7454848),
 ('SigTest', 7454848),
 ('RFiMit', 7454848),
 ('RFIConv', 7454848),
 ('SigDb', 3686528),
 ('lobeSig', 345728),
 ('MultiSig', 115328),
 ('MultiDFT', 115328),
 ('sidelobes', 115328),
 ('noise', 57728),
 ('MDFT_linear', 57728),
 ('MDFT_db', 57728),
 ('test', 55424),
 ('SKarr', 49280),
 ('askDft', 32112),
 ('QpskDft', 16112),
 ('time', 14512),
 ('spectra', 14512),
 ('bfskDft', 9712),
 ('dft', 6512),
 ('_iii', 6355),
 ('_i23', 6355),
 ('_i19', 3197),
 ('_ii', 3099),
 ('_i24', 3099),
 ('_i14', 2494),
 ('_i22', 1945),
 ('_i11', 1728),
 ('RFIconv', 1688),
 ('_i2', 1387),
 ('_i5', 1227),
 ('_i21', 1186),
 ('_i17', 1158),
 ('_i15', 1095),
 ('_i8', 1015),
 ('_i20', 870),
 ('_i', 753),
 ('_i1', 753),
 ('_i25', 753),
 ('_i9', 392),
 ('_i12', 392),
 ('_i6', 337),
 ('_i16', 324),
 ('_i3', 323),
 ('_ih', 312),
 ('In', 312),
 ('_', 296),
 ('_25', 296),
 ('_oh', 

In [ ]:
del ConvMit
del errorCalc
del Sig